# JARVIS V6 — Persistent Colab Studio
Run the AI from source, keep learned state on Google Drive, chat first, then scale and train on GPU. The GitHub repository is the source code; checkpoint `.pt` files are the learned model state.

## 1) Clone the repository

In [ ]:
REPO_URL = 'YOUR_GITHUB_REPOSITORY_URL'
%cd /content
!rm -rf jarvis_v6
!git clone --depth 1 $REPO_URL jarvis_v6
%cd /content/jarvis_v6
!python -m pip install -q -r requirements.txt

## 2) Mount Google Drive for persistent JARVIS state

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
STATE_ROOT = Path('/content/drive/MyDrive/JARVIS_STATE')
STATE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['JARVIS_ROOT'] = str(STATE_ROOT)
print('Persistent state:', STATE_ROOT)

## 3) Import the learned state from GitHub
Download the `jarvis_v6_state.zip` artifact from your completed GitHub Actions run. This archive should contain `checkpoints/*.pt` plus the state/data folders. Upload it once here.

In [ ]:
from google.colab import files
import shutil, zipfile, tempfile
uploads = files.upload()
for name in uploads:
    src = Path('/content') / name
    if zipfile.is_zipfile(src):
        tmp = Path(tempfile.mkdtemp(prefix='jarvis_import_'))
        with zipfile.ZipFile(src) as z: z.extractall(tmp)
        roots = list(tmp.rglob('checkpoints'))
        bundle_root = roots[0].parent if roots else tmp
        for folder in ['data','checkpoints','generations','workspace']:
            candidate = bundle_root / folder
            if candidate.exists(): shutil.copytree(candidate, STATE_ROOT / folder, dirs_exist_ok=True)
        shutil.rmtree(tmp, ignore_errors=True)
        print('Imported:', name)
    else:
        print('Not a zip, skipping:', name)
    src.unlink(missing_ok=True)
print('State ready')

## 4) Verify the checkpoint and GPU

In [ ]:
%cd /content/jarvis_v6
!python main.py status
!python -c "import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print('device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"


## 5) CHAT FIRST
Run the next cell. It launches JARVIS Studio with a chat panel and a live monitor. The Studio is the place where you can talk to the current checkpoint and watch training/evolution state.

In [ ]:
!python main.py studio

## 6) After you have tested chat, scale the model to the Colab profile
This creates a new generation. Compatible learned tensors are copied; newly created capacity is freshly initialized.

In [ ]:
%cd /content/jarvis_v6
!python main.py profile colab
!python main.py status

## 7) Small GPU sanity training run

In [ ]:
!python main.py train --steps 50
!python main.py benchmark

## 8) Architecture evolution test

In [ ]:
!python main.py evolve
!python main.py status
!python main.py manifest

## 9) Public web refresh + learning

In [ ]:
!python main.py crawl
!python main.py train --steps 100
!python main.py benchmark

## 10) Longer autonomous run
This lets the internal policy choose among training, web refresh, evolution, and benchmarking within the configured research sandbox.

In [ ]:
!python main.py autonomous --cycles 30

## 11) Save the learned AI state back to Drive

In [ ]:
!python main.py bundle --destination /content/drive/MyDrive/JARVIS_BUNDLES/jarvis_v6_state

## Checkpoints and logs
- `checkpoints/generation_XXXXXX.pt` = the learned model weights/state
- `generations/.../result.json` = what the accepted/rejected evolution experiment did
- `workspace/experiments.jsonl` = experiment history
- `workspace/events.jsonl` = live training/evolution events
- GitHub Actions logs = what the remote runner printed while executing JARVIS

The runtime is temporary, so Drive persistence is what lets JARVIS resume after Colab restarts.